# OpenAI Responses API

## 模組脈絡：2026 的主線 API——更適合 agent、工具與狀態管理

本筆記隸屬 **05-行為收斂**。**Responses API** 是 OpenAI 2026 建議的新主線入口，特別適合需要內建工具（`file_search`、`web_search`）、多模態輸入、串流事件、狀態接續與 agent 工作流的場景。

需要精準區分：**Chat Completions 仍可用於簡單聊天/相容舊系統**，但新專案若會用工具、RAG、多輪狀態或 agent orchestration，建議優先從 Responses API 開始。**Assistants API 已棄用，官方排定 2026-08-26 sunset**；原本的 assistants / threads / runs 教材應遷移到 Responses API 或 Conversations API。

本筆記示範：基本 Responses 呼叫、用 `file_search` 內建工具做 RAG、用 `previous_response_id` 做簡單多輪接續。

## 0. 環境設定

In [ ]:
from dotenv import load_dotenv
import os
from contextlib import ExitStack

from openai import OpenAI

load_dotenv()

client = OpenAI()  # 讀取 OPENAI_API_KEY
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4-mini")

## 1. 基本 Responses API 呼叫

與 Chat Completions 的差異：用 `input`（不是 `messages`）、用 `response.output_text` 取一般文字結果。若要處理工具呼叫、reasoning、串流事件或多段 content blocks，請檢查完整的 `response.output`。

In [ ]:
response = client.responses.create(
    model=OPENAI_MODEL,
    input="用一句話說明 Responses API 與 Chat Completions 的差別。",
)
print(response.output_text)

### 從 Chat Completions 遷移對照

| | Chat Completions | Responses（新專案建議入口） |
|---|---|---|
| 適合場景 | 簡單聊天、舊系統相容 | 工具、RAG、多模態、狀態接續、agent 工作流 |
| 輸入 | `messages=[...]` | `input="..."` 或 input item 陣列 |
| 取輸出 | `resp.choices[0].message.content` | `resp.output_text`；進階情境看 `resp.output` |
| 多輪 | 自行維護 messages | `previous_response_id`、重送 output items，或 Conversations API |
| 內建工具 | 較有限/需自行整合 | `file_search` / `web_search` 等 hosted tools |
| Assistants 遷移 | 原本的 thread/run 模型 | 改用 Responses + tools；需要長期 conversation 物件時看 Conversations API |

## 2. 建立 Vector Store 並上傳檔案（RAG 知識庫）

Responses 的 `file_search` 工具會自動對 vector store 做檢索。

In [ ]:
# 建立 vector store 並上傳本資料夾既有 PDF（請確認檔案存在）
vector_store = client.vector_stores.create(name="My Knowledge Store")

file_paths = ["1130219.pdf", "1130513.pdf"]

with ExitStack() as stack:
    file_streams = [stack.enter_context(open(path, "rb")) for path in file_paths]
    file_batch = client.vector_stores.file_batches.upload_and_poll(
        vector_store_id=vector_store.id,
        files=file_streams,
    )

print("status:", file_batch.status, "| counts:", file_batch.file_counts)

## 3. 用 `file_search` 內建工具做 RAG

不需要 Assistant/thread/run，一次呼叫即可檢索並作答。

In [ ]:
response = client.responses.create(
    model=OPENAI_MODEL,
    input="知識庫裡的檔案在講什麼？請摘要重點。",
    tools=[{"type": "file_search", "vector_store_ids": [vector_store.id]}],
)
print(response.output_text)

## 4. 多輪對話：用 `previous_response_id` 做簡單接續

`previous_response_id` 適合示範「延續上一輪回答」：伺服器端會參考前一個 response 的上下文。若課程要做可保存、可列舉、可跨工作流管理的長期對話狀態，應改用 **Conversations API** 或在應用層明確保存狀態。

In [ ]:
followup = client.responses.create(
    model=OPENAI_MODEL,
    input="針對上一個回答，再補充一個實務應用例子。",
    previous_response_id=response.id,
    tools=[{"type": "file_search", "vector_store_ids": [vector_store.id]}],
)
print(followup.output_text)

---

## 本章小結

1. **Responses API** 是 2026 新專案建議入口：`input` + `output_text` 起步，進階情境檢查完整 `output`。
2. **Chat Completions 不是本章重點**：它仍可用於簡單聊天或舊系統相容；agent、工具、RAG、多輪狀態請優先用 Responses。
3. **內建工具**：`file_search`（RAG）、`web_search` 等，免自己接檢索三步協定。
4. **多輪狀態**：`previous_response_id` 適合簡單接續；需要長期 conversation 物件時看 Conversations API。
5. **Assistants API（assistants/threads/runs）已棄用，官方排定 2026-08-26 sunset**，新專案請用 Responses API。